## Scraping House Journal Data

The MA House Journal API as defined on the [MA Legislature Website](https://malegislature.gov/api/swagger/index.html?url=/api/swagger/v1/swagger.json#/RollCalls/RollCalls_GetRollCallAll) just does not work. Even the following example curl request fails to locate a journal that clearly exists:

```console
curl -X GET "https://malegislature.gov/api/Documents/hj09172025/RollCalls" -H "accept: application/json"
```

As a result, in order to properly automate scraping the pdf files, we can do the following approach.

A download of a PDF file has the following URL:

https://malegislature.gov/Journal/House/194/RollCalls/hj09172025_RollCalls.pdf

The two most important paramters are "194" and "hj09172025". 194 is the General Court Number and "hj09172025" is the document number. A document number is broken into 2 components

hj - House Journal
09172025 - The date the journal was published.

As a result, we can programmatically check for each valid URL and download a PDF for it. This unfortunately runs into the problem of doing on average # of total General Court sessions * 365 requests per year.

This could be handled by simply manually checking for each link, copying it and downloading through Python. However, for the sake of enrichment and learning, the following code automates getting every possible valid House Journal Roll call vote.

In [6]:
import requests
import pandas as pd
import time
import datetime
import random
import traceback
import os

In [2]:
# Create arrays for each General Court Numbers
court_numbers = [i for i in range(183, 195)]
court_years = [i for i in range(2003, 2026, 2)]
months = [i for i in range(1, 13)]
days = [i for i in range(1, 32)]

In [3]:
# Create a base url to modify
base = 'https://malegislature.gov/Journal/House/'

# Define a chunk_size for reading pdfs
chunk_size = 2000

In [10]:
def scrape_house(use_delay = False):
    try:
        # Loop through each court number
        for i, court_number in enumerate(court_numbers):

            start_year = court_years[i]
            end_year = start_year + 2

            while start_year != end_year and start_year != 2026:
                print(f"Checking for House Roll Call Vote PDFs for General Court {court_number}, year {start_year}")

                # Create a new directory for current year to write pdfs to
                directory_path = f'../../data/HJRollCalls/{start_year}'
                os.makedirs(directory_path, exist_ok=True)

                for month in months:
                    for day in days:

                        # Create a url to request
                        date_str = f'hj{month:02d}{day:02d}{start_year}'
                        url = f'{base}{court_number}/RollCalls/{date_str}_RollCalls.pdf'

                        print(f"[{date_str[2:]}] - Searching for RollCall Vote pdf...")

                        # Add random delay to request
                        if use_delay:
                            time.sleep(random.uniform(1, 3))

                        r = requests.get(url, stream=True)

                        if r.status_code == 200:
                            # Open pdf file to write output to
                            path = f'{directory_path}/{date_str}.pdf'
                            with open(path, 'wb') as f:
                                for chunk in r.iter_content(chunk_size=chunk_size):
                                    f.write(chunk)

                            print("Found Roll Call and wrote to path: ", path)
                        else:
                            print("No roll call for this date!")

                #Increment start year
                start_year += 1

        # No issues occured when parsing
        return True
    except:
        print("An error occured when downloading House Roll Call pdfs")
        traceback.print_exc()
        return False


In [11]:
# Call parsing run
if (scrape_house()):
    # Print success message if parsing worked.
    print("PDFs parsed successfully! Check output in data/HJRollCall subfolders")
else:
    print("An error occurred, check stack trace for more information.")

Checking for House Roll Call Vote PDFs for General Court 183, year 2003
[01012003] - Searching for RollCall Vote pdf...
No roll call for this date!
[01022003] - Searching for RollCall Vote pdf...
No roll call for this date!
[01032003] - Searching for RollCall Vote pdf...
No roll call for this date!
[01042003] - Searching for RollCall Vote pdf...
No roll call for this date!
[01052003] - Searching for RollCall Vote pdf...
No roll call for this date!
[01062003] - Searching for RollCall Vote pdf...
No roll call for this date!
[01072003] - Searching for RollCall Vote pdf...
No roll call for this date!
[01082003] - Searching for RollCall Vote pdf...
Found Roll Call and wrote to path:  ../../data/HJRollCalls/2003/hj01082003.pdf
[01092003] - Searching for RollCall Vote pdf...
Found Roll Call and wrote to path:  ../../data/HJRollCalls/2003/hj01092003.pdf
[01102003] - Searching for RollCall Vote pdf...
No roll call for this date!
[01112003] - Searching for RollCall Vote pdf...
No roll call for t

In [13]:
# Count number of files parsed
count = sum([len(files) for r, d, files in os.walk('../../data/HJRollCalls')])

print(f"Downloaded {count} files from MA Legislature website")

Downloaded 599 files from MA Legislature website
